# Freddie Mac Multi-Year Origination Validation

## Purpose

This notebook applies the reusable Freddie Mac origination pipeline to the
2015–2019 annual samples.

The detailed 2015 EDA established the baseline cleaning and validation
decisions. This notebook checks whether those decisions remain appropriate
for subsequent years and identifies:

- structural or schema problems;
- undocumented categorical codes;
- documented sentinel values;
- changes in missingness and feature ranges;
- disclosure or data-definition changes;
- potential borrower-population changes requiring later drift analysis.

The notebook initially validates 2015 and 2016 and will be extended through
2019 after each annual sample is reviewed.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


# Identify the repository root when the notebook runs from /notebooks.
CURRENT_DIRECTORY = Path.cwd()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.freddie_ingestion import load_origination
from src.freddie_quality import create_origination_quality_report


print("Notebook directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)

Notebook directory: e:\capstone_project\mortgage-model-reliability\notebooks
Project root: e:\capstone_project\mortgage-model-reliability


## 1. File Configuration

Define the annual origination files using paths relative to the project root.
The notebook will stop if an expected input file is missing.

In [2]:
YEARS = [2015, 2016]

RAW_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Freddie_Mac"
)

ORIGINATION_FILES = {
    year: (
        RAW_DATA_DIRECTORY
        / f"sample_{year}"
        / f"sample_orig_{year}.txt"
    )
    for year in YEARS
}


for year, file_path in ORIGINATION_FILES.items():
    print(
        year,
        "| Exists:",
        file_path.exists(),
        "| Path:",
        file_path,
    )

2015 | Exists: True | Path: e:\capstone_project\mortgage-model-reliability\data\raw\Freddie_Mac\sample_2015\sample_orig_2015.txt
2016 | Exists: True | Path: e:\capstone_project\mortgage-model-reliability\data\raw\Freddie_Mac\sample_2016\sample_orig_2016.txt


## 2. Load and Structurally Validate Annual Origination Data

`load_origination()` first validates the raw file structure and then loads the
pipe-delimited records into a pandas DataFrame. At this stage, no cleaning is
performed.

In [3]:
raw_origination = {}
structure_rows = []

for year, file_path in ORIGINATION_FILES.items():
    annual_data = load_origination(file_path)

    raw_origination[year] = annual_data

    structure_report = annual_data.attrs[
        "structure_validation"
    ]

    structure_rows.append(
        {
            "year": year,
            **structure_report,
        }
    )


structure_validation = pd.DataFrame(structure_rows)

display(structure_validation)

,year,rows_scanned,expected_fields_per_row,field_count_errors,field_length_errors
0,2015,50000,31,0,0
1,2016,50000,31,0,0


In [4]:
raw_origination[2015]
raw_origination[2016]

,CLASSIC FICO,FIRST PAYMENT DATE,FIRST TIME HOMEBUYER INDICATOR,MATURITY DATE,METROPOLITAN STATISTICAL AREA (MSA) OR METROPOLITAN DIVISION,MORTGAGE INSURANCE PERCENTAGE (MI %),NUMBER OF UNITS,OCCUPANCY STATUS,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),ORIGINAL DEBT-TO-INCOME (DTI) RATIO,...,ORIGINAL LOAN TERM,NUMBER OF BORROWERS,SELLER NAME,SUPER CONFORMING FLAG,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,HARP INDICATOR,PROPERTY VALUATION METHOD,INTEREST ONLY (I/O) INDICATOR,VANTAGESCORE 4.0
0,686,201604,N,203603,39820.0,0,1,P,65,41,...,240,2,OTHER,N,<NA>,<NA>,N,7,N,9999
1,774,201603,Y,204602,29620.0,30,1,P,95,30,...,360,1,OTHER,N,<NA>,<NA>,N,7,N,9999
2,808,201604,N,203103,30700.0,0,1,P,81,15,...,180,2,OTHER,N,<NA>,<NA>,N,7,N,9999
3,784,201603,N,204602,NaN,0,1,P,82,42,...,360,2,OTHER,N,<NA>,<NA>,N,7,N,9999
4,798,201603,N,204602,47664.0,0,1,P,28,22,...,360,1,OTHER,N,<NA>,<NA>,N,7,N,9999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,746,201612,N,203111,NaN,0,1,P,80,34,...,180,1,OTHER,N,<NA>,<NA>,N,7,N,9999
49996,803,201612,N,203111,NaN,0,1,P,62,18,...,180,2,OTHER,N,<NA>,<NA>,N,7,N,9999
49997,729,201612,N,203111,NaN,0,1,P,75,36,...,180,2,OTHER,N,<NA>,<NA>,N,7,N,9999
49998,748,201612,N,204611,17140.0,0,1,P,97,32,...,360,1,OTHER,N,<NA>,<NA>,N,7,N,9999


In [5]:
for year, data in raw_origination.items():
    print(
        year,
        "| Rows:",
        data.shape[0],
        "| Columns:",
        data.shape[1],
    )

2015 | Rows: 50000 | Columns: 31
2016 | Rows: 50000 | Columns: 31


## 3. Annual Data-Quality Reports

The quality checks do not modify the raw data. They report:

- undocumented categorical codes;
- documented sentinel frequencies;
- raw missingness by feature.

Unexpected codes require investigation before cleaning or interpreting
year-to-year differences as borrower-population drift.

In [6]:
quality_reports = {}

for year, data in raw_origination.items():
    quality_reports[year] = (
        create_origination_quality_report(
            data=data,
            year=year,
        )
    )


unexpected_summary = pd.DataFrame(
    [
        {
            "year": year,
            "unexpected_code_rows": len(
                quality_reports[year]["unexpected_codes"]
            ),
        }
        for year in YEARS
    ]
)

display(unexpected_summary)

,year,unexpected_code_rows
0,2015,0
1,2016,0


### 3.1 Documented Sentinel Frequencies

Sentinels are documented codes representing unavailable, unknown, or
undisclosed information. Differences in sentinel frequency may reflect data
availability or disclosure changes rather than borrower-population drift.

In [7]:
sentinel_counts = pd.concat(
    [
        quality_reports[year]["sentinel_counts"]
        for year in YEARS
    ],
    ignore_index=True,
)


nonzero_sentinels = (
    sentinel_counts.loc[
        sentinel_counts["count"] > 0,
        [
            "year",
            "column",
            "sentinel",
            "count",
            "percentage",
        ],
    ]
    .sort_values(
        by=["column", "year"]
    )
    .reset_index(drop=True)
)


display(nonzero_sentinels)

,year,column,sentinel,count,percentage
0,2015,FIRST TIME HOMEBUYER INDICATOR,9,8,0.016
1,2016,FIRST TIME HOMEBUYER INDICATOR,9,2,0.004
2,2015,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),999,1,0.002
3,2015,ORIGINAL DEBT-TO-INCOME (DTI) RATIO,999,4020,8.040
4,2016,ORIGINAL DEBT-TO-INCOME (DTI) RATIO,999,2528,5.056
5,2015,ORIGINAL LOAN-TO-VALUE (LTV),999,1,0.002
6,2015,PROPERTY VALUATION METHOD,7,50000,100.000
7,2016,PROPERTY VALUATION METHOD,7,50000,100.000
8,2015,VANTAGESCORE 4.0,9999,50000,100.000
9,2016,VANTAGESCORE 4.0,9999,50000,100.000


### 3.2 Raw Missingness Comparison

Raw null values and documented sentinels are reported separately. A field
containing a sentinel is not counted as null in this table.

In [8]:
missingness = pd.concat(
    [
        quality_reports[year]["missingness"]
        for year in YEARS
    ],
    ignore_index=True,
)


missingness_comparison = (
    missingness.pivot(
        index="column",
        columns="year",
        values="missing_percentage",
    )
    .reset_index()
)

missingness_comparison.columns.name = None

missingness_comparison = (
    missingness_comparison.rename(
        columns={
            2015: "missing_2015_pct",
            2016: "missing_2016_pct",
        }
    )
)

missingness_comparison[
    "change_2016_minus_2015_pp"
] = (
    missingness_comparison["missing_2016_pct"]
    - missingness_comparison["missing_2015_pct"]
)


nonzero_missingness = (
    missingness_comparison.loc[
        (
            missingness_comparison["missing_2015_pct"] > 0
        )
        | (
            missingness_comparison["missing_2016_pct"] > 0
        )
    ]
    .sort_values("column")
    .reset_index(drop=True)
)


display(nonzero_missingness)

,column,missing_2015_pct,missing_2016_pct,change_2016_minus_2015_pp
0,METROPOLITAN STATISTICAL AREA (MSA) OR METROPO...,11.208,10.108,-1.100
1,PRE-HARP LOAN SEQUENCE NUMBER,91.966,94.950,2.984
2,SPECIAL ELIGIBILITY PROGRAM,99.350,97.884,-1.466


In [9]:
from src.freddie_cleaning import clean_origination
from src.freddie_validation import validate_origination

## 4. Cleaning and Critical Validation

Cleaning preserves every raw field and appends `_CLEAN` fields. Critical
validation stops processing if the schema, row count, loan identifiers,
cleaned sentinels, or postal-code representation fail.

In [10]:
cleaned_origination = {}
validation_rows = []

for year, raw_data in raw_origination.items():
    cleaned_data = clean_origination(raw_data)

    validation_report = validate_origination(
        raw=raw_data,
        cleaned=cleaned_data,
        expected_rows=50000,
    )

    cleaned_origination[year] = cleaned_data

    validation_rows.append(
        {
            "year": year,
            **validation_report,
        }
    )


validation_summary = pd.DataFrame(validation_rows)

display(validation_summary)

,year,rows,raw_columns,cleaned_columns,missing_loan_ids,duplicate_loan_ids,invalid_postal_codes
0,2015,50000,31,37,0,0,0
1,2016,50000,31,37,0,0,0


In [11]:
from src.freddie_summary import create_annual_summary

## 5. Standardized Annual Summaries

The same summary function is applied to every year so that differences are
comparable. Clean fields are used where documented sentinels were replaced
with missing values.

In [12]:
annual_summaries = {}

for year, cleaned_data in cleaned_origination.items():
    annual_summaries[year] = create_annual_summary(
        data=cleaned_data,
        year=year,
    )


numeric_summary = pd.concat(
    [
        annual_summaries[year]["numeric"]
        for year in YEARS
    ],
    ignore_index=True,
)

categorical_summary = pd.concat(
    [
        annual_summaries[year]["categorical"]
        for year in YEARS
    ],
    ignore_index=True,
)


display(
    numeric_summary.round(2)
)

,year,variable,count,missing,missing_percentage,mean,std,min,q1,median,q3,max
0,2015,FICO,50000.0,0,0.00,749.76,47.51,462.00,718.00,760.00,788.00,834.0
1,2015,DTI,45980.0,4020,8.04,33.84,9.39,1.00,27.00,35.00,42.00,50.0
2,2015,LTV,49999.0,1,0.00,73.51,17.51,5.00,65.00,78.00,83.00,241.0
3,2015,CLTV,49999.0,1,0.00,74.40,18.02,5.00,66.00,79.00,85.00,854.0
4,2015,UPB,50000.0,0,0.00,224006.06,119990.50,10000.00,131000.00,200000.00,299000.00,974000.0
5,2015,INTEREST RATE,50000.0,0,0.00,3.97,0.45,2.38,3.75,4.00,4.25,5.5
6,2015,LOAN TERM,50000.0,0,0.00,315.29,75.92,96.00,240.00,360.00,360.00,366.0
7,2015,MI PERCENTAGE,50000.0,0,0.00,5.90,11.15,0.00,0.00,0.00,0.00,40.0
8,2016,FICO,50000.0,0,0.00,748.79,46.71,456.00,716.00,757.00,787.00,832.0
9,2016,DTI,47472.0,2528,5.06,34.09,9.37,1.00,27.00,35.00,42.00,50.0


### 5.1 Investigate Extreme 2016 LTV and CLTV Values

The central LTV and CLTV distributions are stable, but the maximum value of
336 requires record-level investigation. High values are retained until they
can be classified as valid special-program loans, documented disclosure
values, or data-quality anomalies.

In [13]:
ltv_2016 = cleaned_origination[2016][
    "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN"
]

cltv_2016 = cleaned_origination[2016][
    "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN"
]


print("2016 LTV > 105:", int(ltv_2016.gt(105).sum()))
print("2016 CLTV > 105:", int(cltv_2016.gt(105).sum()))
print("2016 LTV = 336:", int(ltv_2016.eq(336).sum()))
print("2016 CLTV = 336:", int(cltv_2016.eq(336).sum()))

2016 LTV > 105: 148
2016 CLTV > 105: 227
2016 LTV = 336: 1
2016 CLTV = 336: 1


In [14]:
extreme_ratio_records_2016 = (
    cleaned_origination[2016]
    .loc[
        ltv_2016.gt(105) | cltv_2016.gt(105),
        [
            "LOAN IDENTIFIER",
            "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
            "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
            "LOAN PURPOSE",
            "SUPER CONFORMING FLAG",
            "PRE-HARP LOAN SEQUENCE NUMBER",
            "SPECIAL ELIGIBILITY PROGRAM",
            "HARP INDICATOR",
            "ORIGINAL UPB",
            "PROPERTY TYPE",
            "OCCUPANCY STATUS",
        ],
    ]
    .copy()
)


display(
    extreme_ratio_records_2016
    .sort_values(
        by=[
            "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
            "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
        ],
        ascending=False,
    )
    .head(20)
)

,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN,LOAN PURPOSE,SUPER CONFORMING FLAG,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,HARP INDICATOR,ORIGINAL UPB,PROPERTY TYPE,OCCUPANCY STATUS
37415,F16Q30476204,336,336,N,N,A08Q30002418,<NA>,Y,138000,CO,P
8090,F16Q10200259,260,333,N,N,F07Q10243093,<NA>,Y,125000,CO,I
24334,F16Q20409731,222,222,N,N,F09Q20311057,<NA>,Y,334000,SF,P
11179,F16Q10275314,183,183,N,N,F06Q40393669,<NA>,Y,155000,SF,P
19277,F16Q20236155,175,186,N,N,F06Q40317266,<NA>,Y,76000,SF,P
3568,F16Q10089992,161,161,N,N,F06Q30152221,<NA>,Y,67000,SF,P
11807,F16Q10291861,155,170,N,N,F07Q30152312,<NA>,Y,212000,SF,P
9162,F16Q10226493,155,155,N,N,F07Q20027043,<NA>,Y,294000,SF,P
11880,F16Q10293693,155,155,N,N,F07Q20020434,<NA>,Y,96000,SF,P
23178,F16Q20369946,153,153,N,N,F08Q10465752,<NA>,Y,123000,CO,P


In [15]:
high_ratio_mask_2016 = (
    ltv_2016.gt(105)
    | cltv_2016.gt(105)
)

high_ratio_data_2016 = cleaned_origination[2016].loc[
    high_ratio_mask_2016
]


print(
    "Total records with LTV or CLTV > 105:",
    len(high_ratio_data_2016),
)

print(
    "LTV > 105:",
    int(ltv_2016.gt(105).sum()),
)

print(
    "CLTV > 105:",
    int(cltv_2016.gt(105).sum()),
)

print(
    "HARP = Y:",
    int(
        high_ratio_data_2016[
            "HARP INDICATOR"
        ].eq("Y").sum()
    ),
)

print(
    "Pre-HARP sequence present:",
    int(
        high_ratio_data_2016[
            "PRE-HARP LOAN SEQUENCE NUMBER"
        ].notna().sum()
    ),
)

print()
print("Loan-purpose counts:")
print(
    high_ratio_data_2016[
        "LOAN PURPOSE"
    ].value_counts(dropna=False)
)

Total records with LTV or CLTV > 105: 227
LTV > 105: 148
CLTV > 105: 227
HARP = Y: 227
Pre-HARP sequence present: 227

Loan-purpose counts:
LOAN PURPOSE
N    227
Name: count, dtype: int64[pyarrow]


### Extreme-ratio findings for 2016

- A total of 227 loans have LTV or CLTV above 105%.
- All 227 loans are identified as HARP loans.
- All 227 have a populated Pre-HARP Loan Sequence Number.
- All 227 have no-cash-out refinance loan purpose (`N`).
- The maximum LTV and CLTV of 336% are consistent with the special
  high-LTV refinance population represented by HARP.
- These values are preserved and must not be treated as generic range
  errors or replaced with missing values.
- Future range validation should evaluate high ratios together with their
  program indicators rather than applying a universal 105% upper limit.

In [16]:
high_ratio_summary_rows = []

for year, data in cleaned_origination.items():
    annual_ltv = data[
        "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN"
    ]

    annual_cltv = data[
        "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN"
    ]

    high_ratio_mask = (
        annual_ltv.gt(105)
        | annual_cltv.gt(105)
    )

    high_ratio_records = data.loc[high_ratio_mask]

    high_ratio_summary_rows.append(
        {
            "year": year,
            "high_ratio_records": len(
                high_ratio_records
            ),
            "high_ratio_percentage": (
                len(high_ratio_records)
                / len(data)
                * 100
            ),
            "ltv_above_105": int(
                annual_ltv.gt(105).sum()
            ),
            "cltv_above_105": int(
                annual_cltv.gt(105).sum()
            ),
            "harp_y": int(
                high_ratio_records[
                    "HARP INDICATOR"
                ].eq("Y").sum()
            ),
            "pre_harp_id_present": int(
                high_ratio_records[
                    "PRE-HARP LOAN SEQUENCE NUMBER"
                ].notna().sum()
            ),
            "maximum_ltv": annual_ltv.max(),
            "maximum_cltv": annual_cltv.max(),
        }
    )


high_ratio_summary = pd.DataFrame(
    high_ratio_summary_rows
)

display(
    high_ratio_summary.round(3)
)

,year,high_ratio_records,high_ratio_percentage,ltv_above_105,cltv_above_105,harp_y,pre_harp_id_present,maximum_ltv,maximum_cltv
0,2015,530,1.060,361,530,530,530,241.0,854.0
1,2016,227,0.454,148,227,227,227,336.0,336.0


### High-ratio comparison findings

- All high-ratio records in both years are linked to HARP and contain a
  Pre-HARP Loan Sequence Number.
- The high-ratio population decreased from 530 loans (1.060%) in 2015 to
  227 loans (0.454%) in 2016.
- This represents a 0.606 percentage-point reduction in the annual sample's
  HARP-related high-ratio segment.
- The change represents product/program mix variation rather than malformed
  data or a schema difference.
- The 2015 maximum CLTV of 854 and the 2016 maximum LTV/CLTV of 336 are
  retained because their records have consistent HARP indicators.
- The reusable quality pipeline now flags LTV or CLTV above 105% when the
  record is not supported by both a HARP indicator and a populated Pre-HARP
  Loan Sequence Number.
- Applying this rule produced zero warnings for both 2015 and 2016.

In [17]:
for year in YEARS:
    high_ratio_warnings = quality_reports[
        year
    ]["high_ratio_warnings"]

    print(
        year,
        "| Unsupported high-ratio records:",
        len(high_ratio_warnings),
    )

    display(high_ratio_warnings)

2015 | Unsupported high-ratio records: 0


,year,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,HIGH LTV FLAG,HIGH CLTV FLAG


2016 | Unsupported high-ratio records: 0


,year,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,HIGH LTV FLAG,HIGH CLTV FLAG


Before connecting this to the pipeline, confirm the rule can actually detect an unsupported record the rule can actually detect an unsupported record

In [18]:
quality_rule_test = raw_origination[2016].iloc[
    [0]
].copy()

quality_rule_test[
    "ORIGINAL LOAN-TO-VALUE (LTV)"
] = 120

quality_rule_test[
    "HARP INDICATOR"
] = "N"

quality_rule_test[
    "PRE-HARP LOAN SEQUENCE NUMBER"
] = pd.NA


test_quality_report = create_origination_quality_report(
    data=quality_rule_test,
    year=2016,
)

display(
    test_quality_report[
        "high_ratio_warnings"
    ]
)

,year,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,HIGH LTV FLAG,HIGH CLTV FLAG
0,2016,F16Q10000006,120,65,N,<NA>,True,False


## 6. Categorical Distribution Comparison

Category percentages are aligned across years. The largest percentage-point
changes are reviewed as possible population or product-mix changes, not as
formal statistical drift conclusions.

In [19]:
categorical_comparison = (
    categorical_summary.pivot_table(
        index=["variable", "category"],
        columns="year",
        values="percentage",
        fill_value=0,
    )
    .reset_index()
)

categorical_comparison.columns.name = None

categorical_comparison = (
    categorical_comparison.rename(
        columns={
            2015: "percentage_2015",
            2016: "percentage_2016",
        }
    )
)

categorical_comparison[
    "change_2016_minus_2015_pp"
] = (
    categorical_comparison["percentage_2016"]
    - categorical_comparison["percentage_2015"]
)

categorical_comparison[
    "absolute_change_pp"
] = categorical_comparison[
    "change_2016_minus_2015_pp"
].abs()


largest_categorical_changes = (
    categorical_comparison
    .sort_values(
        "absolute_change_pp",
        ascending=False,
    )
    .head(20)
    .reset_index(drop=True)
)


display(
    largest_categorical_changes.round(3)
)

,variable,category,percentage_2015,percentage_2016,change_2016_minus_2015_pp,absolute_change_pp
0,LOAN PURPOSE,C,20.666,22.578,1.912,1.912
1,NUMBER OF BORROWERS,2,51.080,49.436,-1.644,1.644
2,NUMBER OF BORROWERS,1,48.920,50.564,1.644,1.644
3,LOAN PURPOSE,N,35.026,33.478,-1.548,1.548
4,OCCUPANCY STATUS,P,88.192,88.844,0.652,0.652
5,PROPERTY TYPE,SF,65.132,64.558,-0.574,0.574
6,FIRST TIME HOMEBUYER,Y,15.580,16.088,0.508,0.508
7,FIRST TIME HOMEBUYER,N,84.404,83.908,-0.496,0.496
8,PROPERTY STATE,CA,15.576,15.984,0.408,0.408
9,PROPERTY STATE,CO,3.420,3.804,0.384,0.384


### Categorical comparison findings

- No categorical percentage changed by more than 2 percentage points.
- Cash-out refinance loans increased by 1.912 percentage points, while
  no-cash-out refinance loans decreased by 1.548 percentage points.
- One-borrower loans increased by 1.644 percentage points, with an equal
  decrease in two-borrower loans.
- Primary-residence occupancy increased by 0.652 percentage points.
- Changes in property type and individual state representation were smaller
  than 0.6 percentage points.
- These descriptive changes may represent borrower or product-mix variation,
  but they are not yet evidence of statistically significant drift.
- Formal drift metrics and significance testing will be performed in the
  later drift-analysis stage.

In [20]:
from src.freddie_pipeline import run_origination_pipeline

## 7. Save Verified 2016 Outputs

After the quality findings have been reviewed, the end-to-end pipeline is run
to save the cleaned loan-level dataset and its validation, summary, and
quality reports.

In [21]:
PROCESSED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Freddie_Mac"
)


pipeline_result_2016 = run_origination_pipeline(
    year=2016,
    input_file=ORIGINATION_FILES[2016],
    output_directory=PROCESSED_DATA_DIRECTORY,
    expected_rows=50000,
)


print("Structure validation:")
print(
    pipeline_result_2016[
        "structure_validation"
    ]
)

print()
print("Data validation:")
print(
    pipeline_result_2016[
        "validation"
    ]
)

print()
print("Saved outputs:")

for output_name, output_path in (
    pipeline_result_2016["output_paths"].items()
):
    print(
        f"{output_name}: {output_path}"
    )

Structure validation:
{'rows_scanned': 50000, 'expected_fields_per_row': 31, 'field_count_errors': 0, 'field_length_errors': 0}

Data validation:
{'rows': 50000, 'raw_columns': 31, 'cleaned_columns': 37, 'missing_loan_ids': 0, 'duplicate_loan_ids': 0, 'invalid_postal_codes': 0}

Saved outputs:
cleaned: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2016_clean.parquet
numeric_summary: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2016_numeric_summary.csv
categorical_summary: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2016_categorical_summary.csv
unexpected_codes: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2016_unexpected_codes.csv
sentinel_counts: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2016_sentinel_counts.csv
missingness: e:\capstone_project\mortgage-model-reliability\data\proc

Read back the saved high-ratio warning CSV and validation JSON to confirm
that the program-aware quality results were written correctly and that the
warning counts agree across both out

In [23]:
import json


saved_high_ratio_warnings_2016 = pd.read_csv(
    pipeline_result_2016[
        "output_paths"
    ]["high_ratio_warnings"]
)


with pipeline_result_2016[
    "output_paths"
]["validation"].open(
    mode="r",
    encoding="utf-8",
) as validation_file:
    saved_validation_2016 = json.load(
        validation_file
    )


print(
    "Saved high-ratio warning rows:",
    len(saved_high_ratio_warnings_2016),
)

print(
    "Validation JSON warning rows:",
    saved_validation_2016[
        "high_ratio_warning_rows"
    ],
)

display(saved_high_ratio_warnings_2016)

Saved high-ratio warning rows: 0
Validation JSON warning rows: 0


,year,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,HIGH LTV FLAG,HIGH CLTV FLAG


## 8. Saved-Output Verification

The saved Parquet dataset is read back and validated again to confirm that
serialization preserved the expected rows, columns, cleaned fields, loan
identifiers, and postal-code representation.

In [22]:
saved_cleaned_2016 = pd.read_parquet(
    pipeline_result_2016[
        "output_paths"
    ]["cleaned"]
)


readback_validation_2016 = validate_origination(
    raw=raw_origination[2016],
    cleaned=saved_cleaned_2016,
    expected_rows=50000,
)


print(
    "Saved shape:",
    saved_cleaned_2016.shape,
)

print(
    "Postal-code dtype:",
    saved_cleaned_2016[
        "POSTAL CODE_CLEAN"
    ].dtype,
)

print(
    "Read-back validation:",
    readback_validation_2016,
)

Saved shape: (50000, 37)
Postal-code dtype: string
Read-back validation: {'rows': 50000, 'raw_columns': 31, 'cleaned_columns': 37, 'missing_loan_ids': 0, 'duplicate_loan_ids': 0, 'invalid_postal_codes': 0}


## 9. Conclusion

The reusable Freddie Mac origination pipeline was successfully applied to
the 2016 annual sample.

### Validation outcome

- 50,000 raw records were processed.
- Every raw record contained the expected 31 fields.
- No field-count or maximum-length violations were detected.
- No undocumented categorical codes were detected.
- Cleaning preserved all 31 raw fields and appended six clean fields.
- No missing or duplicate loan identifiers were found.
- No invalid cleaned postal-code values were found.
- The saved Parquet file was successfully read back with 50,000 rows and
  37 columns.
- The program-aware high-ratio quality rule produced zero warnings for 2016,
  and the warning report was successfully saved as a separate CSV file.
### Year-over-year observations

- DTI sentinel usage decreased from 8.040% in 2015 to 5.056% in 2016.
- Raw missingness changes were limited to MSA, Pre-HARP Loan Sequence
  Number, and Special Eligibility Program.
- Numeric central distributions were generally stable.
- Median Original UPB increased by $15,000.
- Mean Original Interest Rate decreased by approximately 0.18 percentage
  points.
- No primary categorical percentage changed by more than 2 percentage
  points.
- The HARP-related high-ratio population decreased from 1.060% to 0.454%.
- All 2015 and 2016 records with LTV or CLTV above 105% were supported by
  both a HARP indicator and a Pre-HARP Loan Sequence Number.

The 2016 sample passes the current quality and validation requirements.
Observed descriptive differences will be evaluated formally during the
later drift-analysis stage.